### Ollama 설치 및 Modelfile 설정하기

In [ ]:
# %pip install langchain-ollama

In [5]:
import os

from langchain_teddynote import logging
from langchain_teddynote.messages import stream_response
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_ollama import ChatOllama

load_dotenv()
logging.langsmith("test0914")

llm = ChatOllama(model="EEVE-Korean-10.8B:latest")


print("OpenAI 키 로드됨 : ", bool(os.getenv("OPENAI_API_KEY")))
print("Google 키 로드됨 : ", bool(os.getenv("GOOGLE_API_KEY")))
print("LangSmith 키 로드됨 : ", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914
OpenAI 키 로드됨 :  True
Google 키 로드됨 :  True
LangSmith 키 로드됨 :  True
LangSmith 프로젝트 :  test0914


In [6]:
prompt = ChatPromptTemplate.from_template("{topic} 에 대하여 간략히 설명해줘")

chain = prompt | llm | StrOutputParser()

answer = chain.stream({"topic":"deep learning"})

stream_response(answer)

딥 러닝은 머신 러닝의 한 분야로, 컴퓨터가 인간의 두뇌가 정보를 처리하고 결정에 도달하는 방식을 모방하여 대량의 데이터로부터 학습하는 데 중점을 둡니다. 딥 러닝 모델은 대량의 데이터를 처리하기 위해 복잡한 데이터 처리 및 학습 알고리즘을 사용하여, 이미지, 음성, 텍스트를 인식하고 분류하는 작업과 같은 다양한 작업에서 정확도를 높일 수 있습니다.

딥 러닝 모델은 인공 신경망(ANNs)이라고 불리는 계층화된 신경망으로 구성됩니다. ANNs는 여러 층의 노드로 이루어져 있으며, 각 노드는 입력값을 받아 새로운 출력값을 계산합니다. 노드 간의 연결은 가중치라고 불리며, 학습 동안 데이터 세트를 기반으로 조정됩니다. 딥 러닝 모델은 많은 층의 노드를 포함하고 있어 데이터에서 복잡한 패턴과 관계를 더 잘 파악할 수 있습니다.

딥 러닝 모델을 훈련시키기 위해 일반적으로 대량의 레이블이 붙은 데이터가 사용됩니다. 이 데이터에서 모델은 입력(예: 이미지, 음성, 텍스트)의 특징을 학습하고, 학습된 특징을 기반으로 한 레이블(예: 카테고리, 감정)을 예측하는 방법을 배웁니다. 모델이 입력과 레이블을 기반으로 오류를 최소화하기 위해 가중치를 조정하면서 학습합니다.

딥 러닝 모델의 성능은 사용되는 특정 알고리즘과 데이터셋에 따라 달라집니다. 딥 러닝은 음성 인식, 이미지 인식, 자연어 처리와 같은 다양한 분야에서 주목할 만한 결과를 얻었습니다.

요약하자면, 딥 러닝은 대량의 데이터로부터 학습하기 위해 인공 신경망(ANNs)을 사용하는 머신 러닝의 한 분야로, 이미지, 음성, 텍스트와 같은 다양한 데이터에서 복잡한 패턴을 이해하고 높은 정확도로 작업을 수행하는 데 사용됩니다.

In [7]:
llm2 = ChatOllama(
    model="gemma:7b",
    format="json",
    temperature=0
)

In [9]:
prompt2 = "유럽 여행지 10곳을 알려주세요. key: 'place'.response in JSON format."

response = llm2.invoke(prompt2)

print(response.content)

{
 "key": "place",
 "response": [
  {
   "place": "프랑스"
  },
  {
   "place": "영국"
  },
  {
   "place": "독일"
  },
  {
   "place": "스페인"
  },
  {
   "place": "이탈리아"
  },
  {
   "place": "네덜란드"
  },
  {
   "place": "폴란드"
  },
  {
   "place": "체코슬로바키아"
  },
  {
   "place": "스웨덴"
  },
  {
   "place": "핀란드"
  }
 ]
}


In [ ]:
import base64
from io import BytesIO

from IPython.display import HTML, display
from PIL import Image
from langchain_core.messages import HumanMessage

def convert_to_base64(pil_image):
    """
    PIL 이미지를 Base64로 인코딩된 문자열로 변환합니다.
    
    :param pil_image: PIL 이미지
    :return: 크기 조정된 Base64 문자열
    """

    buffered = BytesIO()
    pil_image.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return img_str


def plt_img_base64(img_base64):
    """
    Base64로 인코딩된 문자열을 이미지로 표시합니다.
    
    :param img_base64: Base64 문자열
    """

    # Base64 문자열을 소스로 사용하여 HTML img 태그 생성
    image_html = f'<img src="data:image/jpeg;base64,{img_base64}" />'
    display(HTML(image_html))

def prompt_func(data):
    text = data["text"]
    image = data['image']

    image_part = {
        "type": "image_url",
        "image_url": f"data:image/jpeg;base64,{image}"
    }

    content_parts = []

    text_part = {"type": "text", "text":text}

    content_parts.append(image_part)
    content_parts.append(text_part)

    return [HumanMessage(content=content_parts)]

file_path = "jeju-beach.jpg"
pil_image = Image.open(file_path)

image_b64 = convert_to_base64(pil_image)

plt_img_base64(image_b64)

In [13]:
llm3 = ChatOllama(model="llava:7b", temperature=0)

chain = prompt_func | llm3 | StrOutputParser()

query_chain = chain.invoke(
    {"text":"Describe a picture in bullet points", "image": image_b64}
)

print(query_chain)

 - The image is a panoramic photograph of a tropical beach scene.
- It features a clear blue sky and calm turquoise waters.
- The beach is lined with white sand and is dotted with small rocks.
- There is a small island with a lush green top in the center of the image.
- The water is shallow near the shore, with a sandbar visible.
- The beach is surrounded by a coral reef, which is visible under the water's surface.
- The vegetation on the island is lush and green, indicating a healthy ecosystem.
- The image is taken from a distance, providing a wide view of the beach and the island.
- There are no visible texts or distinctive brands in the image.
- The overall impression is one of a serene and idyllic tropical location. 
